# Advanced Python Set Operations: Problems with Solutions

This notebook develops practical mastery of Python sets through progressively harder problems.

## Topics covered

- Cardinality with `len`
- Fast membership testing with `in` and `not in`
- Adding values with `add` and `update`
- Removing values with `remove`, `discard`, `pop`, and `clear`
- Union, intersection, difference, and symmetric difference
- Subset, superset, and disjointness checks
- Hashability and immutable `frozenset` values
- Safe mutation patterns
- Deduplication, indexing, graph edges, permissions, reconciliation, and caching
- Complexity, timing, and memory trade-offs

Every problem includes a task, a complete solution, tests, and an explanation.

## Best-practice checklist

- Use a set when uniqueness or fast membership testing is the main requirement.
- Do not rely on set iteration order.
- Store only hashable values in sets.
- Prefer `discard` when a missing value is acceptable.
- Prefer `remove` when a missing value indicates a bug or invalid state.
- Use set algebra instead of manual loops when it clearly expresses the intent.
- Avoid mutating a set while iterating over that same set.
- Use `frozenset` when a set itself must be hashable.
- Benchmark realistic workloads; do not assume a set is always the best structure.
- Remember that `sys.getsizeof` and `__sizeof__` do not recursively include referenced objects.

## Setup

In [1]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from functools import lru_cache
from itertools import combinations
from statistics import median
from timeit import repeat
import random
import sys

## Compact reference: common set operations

Assertions make the notebook an executable specification.

In [2]:
values = {10, 20, 30}

assert len(values) == 3
assert 20 in values
assert 99 not in values

values.add(40)
values.update([40, 50, 60])
assert values == {10, 20, 30, 40, 50, 60}

values.remove(10)
values.discard(999)
assert values == {20, 30, 40, 50, 60}

removed = values.pop()
assert removed not in values

values.clear()
assert values == set()

print('Reference checks passed.')

Reference checks passed.


# Problem 1 — Stable deduplication

## Task

Remove duplicates while preserving the first occurrence of each value.

A plain `set(iterable)` removes duplicates but does not express the required output order. Use a set only for membership tracking and a list for ordered output.

In [3]:
def unique_in_order(items):
    """Return first occurrences in input order."""
    seen = set()
    result = []

    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result


data = ['red', 'blue', 'red', 'green', 'blue', 'yellow']
expected = ['red', 'blue', 'green', 'yellow']

assert unique_in_order(data) == expected
assert unique_in_order([]) == []
assert unique_in_order([1, 1, 1]) == [1]
print(unique_in_order(data))

['red', 'blue', 'green', 'yellow']


Expected time is `O(n)` and additional space is `O(k)`, where `k` is the number of distinct values.

# Problem 2 — Duplicate report

## Task

Return the first value whose second occurrence is encountered, the index of that occurrence, the set of all duplicated values, and the number of distinct values.

In [4]:
def duplicate_report(items):
    seen = set()
    duplicates = set()
    first_repeat = None

    for index, item in enumerate(items):
        if item in seen:
            duplicates.add(item)
            if first_repeat is None:
                first_repeat = (item, index)
        else:
            seen.add(item)

    return {
        'first_repeat': first_repeat,
        'duplicates': duplicates,
        'distinct_count': len(seen),
    }


report = duplicate_report(['a', 'b', 'c', 'b', 'd', 'a', 'a'])
assert report['first_repeat'] == ('b', 3)
assert report['duplicates'] == {'a', 'b'}
assert report['distinct_count'] == 4
print(report)

{'first_repeat': ('b', 3), 'duplicates': {'b', 'a'}, 'distinct_count': 4}


# Problem 3 — Validate requested permissions

## Task

Validate that every requested permission is known, every required permission is present, and forbidden permissions are absent. Return a structured report.

In [5]:
def validate_permissions(requested, *, allowed, required=frozenset(), forbidden=frozenset()):
    requested = set(requested)
    allowed = set(allowed)
    required = set(required)
    forbidden = set(forbidden)

    unknown = requested - allowed
    missing_required = required - requested
    forbidden_present = requested & forbidden

    return {
        'valid': not unknown and not missing_required and not forbidden_present,
        'unknown': unknown,
        'missing_required': missing_required,
        'forbidden_present': forbidden_present,
    }


result = validate_permissions(
    {'read', 'write', 'admin', 'delete'},
    allowed={'read', 'write', 'export', 'admin'},
    required={'read'},
    forbidden={'admin'},
)

assert result == {
    'valid': False,
    'unknown': {'delete'},
    'missing_required': set(),
    'forbidden_present': {'admin'},
}
print(result)

{'valid': False, 'unknown': {'delete'}, 'missing_required': set(), 'forbidden_present': {'admin'}}


The expressions `requested - allowed`, `required - requested`, and `requested & forbidden` directly encode the business rules.

# Problem 4 — Configuration drift

## Task

Compare expected and observed feature sets. Report missing, unexpected, and all drifted features.

In [6]:
def configuration_drift(expected, observed):
    expected = set(expected)
    observed = set(observed)

    missing = expected - observed
    unexpected = observed - expected
    drift = expected ^ observed

    assert drift == missing | unexpected

    return {
        'in_sync': not drift,
        'missing': missing,
        'unexpected': unexpected,
        'drift': drift,
    }


report = configuration_drift(
    {'logging', 'metrics', 'tracing', 'backups'},
    {'logging', 'metrics', 'debug_toolbar'},
)

assert report['missing'] == {'tracing', 'backups'}
assert report['unexpected'] == {'debug_toolbar'}
assert report['drift'] == {'tracing', 'backups', 'debug_toolbar'}
print(report)

{'in_sync': False, 'missing': {'tracing', 'backups'}, 'unexpected': {'debug_toolbar'}, 'drift': {'backups', 'debug_toolbar', 'tracing'}}


# Problem 5 — Compare many environments

## Task

Find features enabled everywhere, somewhere, and inconsistently across multiple environments.

In [7]:
def compare_environments(environment_features):
    if not environment_features:
        return {'everywhere': set(), 'somewhere': set(), 'inconsistent': set()}

    feature_sets = [set(features) for features in environment_features.values()]
    everywhere = set.intersection(*feature_sets)
    somewhere = set.union(*feature_sets)

    return {
        'everywhere': everywhere,
        'somewhere': somewhere,
        'inconsistent': somewhere - everywhere,
    }


environments = {
    'development': {'logging', 'metrics', 'debug'},
    'staging': {'logging', 'metrics', 'tracing'},
    'production': {'logging', 'metrics', 'tracing', 'backups'},
}

comparison = compare_environments(environments)
assert comparison['everywhere'] == {'logging', 'metrics'}
assert comparison['somewhere'] == {'logging', 'metrics', 'debug', 'tracing', 'backups'}
assert comparison['inconsistent'] == {'debug', 'tracing', 'backups'}
print(comparison)

{'everywhere': {'metrics', 'logging'}, 'somewhere': {'backups', 'logging', 'tracing', 'debug', 'metrics'}, 'inconsistent': {'tracing', 'backups', 'debug'}}


# Problem 6 — Inverted index and keyword search

## Task

Build an inverted index mapping normalized words to document-ID sets. Implement `match_all`, `match_any`, and `match_none` query results.

In [8]:
def tokenize(text):
    return {
        token.strip('.,!?;:()[]{}"\'').lower()
        for token in text.split()
        if token.strip('.,!?;:()[]{}"\'')
    }


def build_inverted_index(documents):
    index = defaultdict(set)
    for document_id, text in documents.items():
        for word in tokenize(text):
            index[word].add(document_id)
    return dict(index)


def search_documents(index, all_document_ids, query_terms):
    terms = {term.lower() for term in query_terms}
    all_document_ids = set(all_document_ids)
    postings = [index.get(term, set()) for term in terms]

    match_all = set.intersection(*postings) if postings else all_document_ids.copy()
    match_any = set.union(*postings) if postings else set()

    return {
        'match_all': match_all,
        'match_any': match_any,
        'match_none': all_document_ids - match_any,
    }


documents = {
    101: 'Python sets support fast membership tests.',
    102: 'Python dictionaries and sets use hashing.',
    103: 'Lists preserve order and permit duplicates.',
    104: 'Hashing supports fast dictionary lookup.',
}

index = build_inverted_index(documents)
result = search_documents(index, documents, {'python', 'sets'})

assert result['match_all'] == {101, 102}
assert result['match_any'] == {101, 102}
assert result['match_none'] == {103, 104}
print("Postings for 'sets':", index['sets'])
print(result)

Postings for 'sets': {101, 102}
{'match_all': {101, 102}, 'match_any': {101, 102}, 'match_none': {104, 103}}


# Problem 7 — Safe removal while filtering

## Task

Remove revoked session IDs. Demonstrate a non-mutating expression, an in-place update, and safe filtering patterns.

In [9]:
active_sessions = {'s1', 's2', 's3', 's4', 's5'}
revoked_sessions = {'s2', 's5', 's9'}

clean_copy = active_sessions - revoked_sessions
assert clean_copy == {'s1', 's3', 's4'}
assert active_sessions == {'s1', 's2', 's3', 's4', 's5'}

active_sessions.difference_update(revoked_sessions)
assert active_sessions == {'s1', 's3', 's4'}
print(active_sessions)

{'s3', 's4', 's1'}


In [10]:
# Do not mutate a set while iterating directly over it.

sessions = {'s1', 's2', 'x1', 'x2'}

# Safe option 1: build a new set.
sessions = {session_id for session_id in sessions if not session_id.startswith('s')}
assert sessions == {'x1', 'x2'}

# Safe option 2: iterate over a snapshot.
sessions = {'s1', 's2', 'x1', 'x2'}
for session_id in sessions.copy():
    if session_id.startswith('s'):
        sessions.remove(session_id)

assert sessions == {'x1', 'x2'}
print(sessions)

{'x1', 'x2'}


# Problem 8 — Strict versus forgiving deletion

## Task

Use `remove` when absence is an error and `discard` when absence is acceptable.

In [11]:
def unregister_strict(registry, name):
    registry.remove(name)


def unregister_if_present(registry, name):
    registry.discard(name)


registry = {'alpha', 'beta', 'gamma'}
unregister_strict(registry, 'beta')
assert registry == {'alpha', 'gamma'}

unregister_if_present(registry, 'missing')
assert registry == {'alpha', 'gamma'}

try:
    unregister_strict(registry, 'missing')
except KeyError as exc:
    assert exc.args == ('missing',)
else:
    raise AssertionError('Expected KeyError')

print(registry)

{'alpha', 'gamma'}


# Problem 9 — Consume an unordered work set

## Task

Process all pending jobs when order does not matter. `set.pop()` removes an arbitrary element.

In [12]:
def process_unordered_jobs(pending_jobs, handler):
    pending_jobs = set(pending_jobs)
    processed = []

    while pending_jobs:
        job = pending_jobs.pop()
        handler(job)
        processed.append(job)

    return processed


handled = []
processed = process_unordered_jobs({'job-A', 'job-B', 'job-C'}, handled.append)

assert set(processed) == {'job-A', 'job-B', 'job-C'}
assert set(handled) == {'job-A', 'job-B', 'job-C'}
assert len(processed) == 3
print('Processed in arbitrary order:', processed)

Processed in arbitrary order: ['job-A', 'job-B', 'job-C']


Use `deque` for FIFO/LIFO work or `heapq` for priority work. A set can still prevent duplicate scheduling.

# Problem 10 — Longest substring without repeated characters

## Task

Use a sliding window and a set of characters currently in the window.

In [13]:
def longest_unique_substring(text):
    left = 0
    window = set()
    best_start = 0
    best_length = 0

    for right, character in enumerate(text):
        while character in window:
            window.remove(text[left])
            left += 1

        window.add(character)
        current_length = right - left + 1

        if current_length > best_length:
            best_start = left
            best_length = current_length

    return text[best_start:best_start + best_length]


assert longest_unique_substring('abcabcbb') == 'abc'
assert longest_unique_substring('bbbbb') == 'b'
assert longest_unique_substring('pwwkew') == 'wke'
assert longest_unique_substring('') == ''
print(longest_unique_substring('set operations'))

perations


Each character enters and leaves the active set at most once, so expected time is `O(n)`.

# Problem 11 — Unique undirected graph edges

## Task

Normalize `(u, v)` and `(v, u)` to one canonical edge and reject self-loops.

In [14]:
def normalize_undirected_edges(edges):
    normalized = set()

    for left, right in edges:
        if left == right:
            continue
        normalized.add(tuple(sorted((left, right))))

    return normalized


raw_edges = [
    ('A', 'B'), ('B', 'A'), ('A', 'C'),
    ('C', 'A'), ('B', 'B'), ('B', 'C'),
]

edges = normalize_undirected_edges(raw_edges)
assert edges == {('A', 'B'), ('A', 'C'), ('B', 'C')}
print(edges)

{('A', 'B'), ('A', 'C'), ('B', 'C')}


# Problem 12 — Detect graph triangles

## Task

For each edge `(u, v)`, common neighbors are `neighbors[u] & neighbors[v]`. Return every triangle once.

In [15]:
def build_adjacency(edges):
    adjacency = defaultdict(set)
    for left, right in edges:
        if left != right:
            adjacency[left].add(right)
            adjacency[right].add(left)
    return dict(adjacency)


def find_triangles(edges):
    adjacency = build_adjacency(edges)
    triangles = set()

    for left, right in normalize_undirected_edges(edges):
        for common_neighbor in adjacency[left] & adjacency[right]:
            triangles.add(tuple(sorted((left, right, common_neighbor))))

    return triangles


graph_edges = [
    ('A', 'B'), ('B', 'C'), ('C', 'A'),
    ('B', 'D'), ('C', 'D'), ('D', 'E'),
]

triangles = find_triangles(graph_edges)
assert triangles == {('A', 'B', 'C'), ('B', 'C', 'D')}
print(triangles)

{('B', 'C', 'D'), ('A', 'B', 'C')}


# Problem 13 — Freeze nested mutable data

## Task

Convert lists, dictionaries, and sets into hashable equivalents so nested records can be deduplicated.

In [16]:
def freeze(value):
    if isinstance(value, dict):
        return frozenset((freeze(key), freeze(item)) for key, item in value.items())
    if isinstance(value, (list, tuple)):
        return tuple(freeze(item) for item in value)
    if isinstance(value, set):
        return frozenset(freeze(item) for item in value)
    return value


records = [
    {'name': 'Ada', 'skills': ['python', 'math']},
    {'skills': ['python', 'math'], 'name': 'Ada'},
    {'name': 'Grace', 'skills': ['compilers']},
]

frozen_records = {freeze(record) for record in records}
assert len(frozen_records) == 2
assert freeze(records[0]) == freeze(records[1])
print(frozen_records)

{frozenset({('name', 'Ada'), ('skills', ('python', 'math'))}), frozenset({('name', 'Grace'), ('skills', ('compilers',))})}


This is structural deduplication. The simple representation does not retain enough type metadata for perfect reconstruction.

# Problem 14 — `frozenset` as a cache key

## Task

Equivalent rule groups should share one cached compiled result, regardless of input order.

In [17]:
@lru_cache(maxsize=None)
def compile_rules(rule_set):
    return tuple(sorted(rule_set))


def get_compiled_rules(rules):
    return compile_rules(frozenset(rules))


first = get_compiled_rules(['trim', 'lowercase', 'deduplicate'])
second = get_compiled_rules(['deduplicate', 'trim', 'lowercase'])

assert first == ('deduplicate', 'lowercase', 'trim')
assert first is second
assert compile_rules.cache_info().hits >= 1
print(first)
print(compile_rules.cache_info())

('deduplicate', 'lowercase', 'trim')
CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)


# Problem 15 — Correct hashing for custom objects

## Task

Equality should depend only on `user_id`. Implement equality and hashing consistently on an immutable class.

In [18]:
@dataclass(frozen=True, eq=False)
class StableUserIdentity:
    user_id: int
    display_name: str

    def __eq__(self, other):
        if not isinstance(other, StableUserIdentity):
            return NotImplemented
        return self.user_id == other.user_id

    def __hash__(self):
        return hash(self.user_id)


users = {
    StableUserIdentity(1, 'Ada'),
    StableUserIdentity(1, 'Ada Lovelace'),
    StableUserIdentity(2, 'Grace'),
}

assert len(users) == 2
assert StableUserIdentity(1, 'Any label') in users
print(users)

{StableUserIdentity(user_id=1, display_name='Ada'), StableUserIdentity(user_id=2, display_name='Grace')}


If `a == b`, then `hash(a) == hash(b)` must also be true. Hash-participating fields should not change while an object is stored in a set.

# Problem 16 — Reconcile transaction feeds

## Task

Report exact matches, internal-only records, external-only records, and shared IDs with conflicting amounts.

In [19]:
def reconcile_transactions(internal_records, external_records):
    internal = set(internal_records)
    external = set(external_records)

    internal_by_id = {transaction_id: amount for transaction_id, amount in internal}
    external_by_id = {transaction_id: amount for transaction_id, amount in external}
    shared_ids = set(internal_by_id) & set(external_by_id)

    return {
        'exact_matches': internal & external,
        'internal_only': internal - external,
        'external_only': external - internal,
        'conflicting_ids': {
            transaction_id
            for transaction_id in shared_ids
            if internal_by_id[transaction_id] != external_by_id[transaction_id]
        },
    }


internal_feed = {('T1', 5000), ('T2', 1250), ('T3', 900)}
external_feed = {('T1', 5000), ('T2', 1300), ('T4', 700)}

result = reconcile_transactions(internal_feed, external_feed)
assert result['exact_matches'] == {('T1', 5000)}
assert result['conflicting_ids'] == {'T2'}
assert ('T3', 900) in result['internal_only']
assert ('T4', 700) in result['external_only']
print(result)

{'exact_matches': {('T1', 5000)}, 'internal_only': {('T3', 900), ('T2', 1250)}, 'external_only': {('T2', 1300), ('T4', 700)}, 'conflicting_ids': {'T2'}}


# Problem 17 — Verify pairwise disjoint groups

## Task

Identify every pair of groups that overlaps and list the shared elements.

In [20]:
def find_group_overlaps(groups):
    overlaps = []

    for (left_name, left_values), (right_name, right_values) in combinations(groups.items(), 2):
        left_set = set(left_values)
        right_set = set(right_values)

        if not left_set.isdisjoint(right_set):
            overlaps.append({
                'groups': (left_name, right_name),
                'shared': left_set & right_set,
            })

    return overlaps


teams = {
    'red': {'Ada', 'Linus', 'Margaret'},
    'blue': {'Grace', 'Edsger'},
    'green': {'Margaret', 'Guido'},
}

overlaps = find_group_overlaps(teams)
assert overlaps == [{'groups': ('red', 'green'), 'shared': {'Margaret'}}]
print(overlaps)

[{'groups': ('red', 'green'), 'shared': {'Margaret'}}]


# Problem 18 — Idempotent event registry

## Task

Register events exactly once and return whether each operation changed the registry.

In [21]:
class EventRegistry:
    def __init__(self):
        self._event_ids = set()

    def register(self, event_id):
        old_size = len(self._event_ids)
        self._event_ids.add(event_id)
        return len(self._event_ids) > old_size

    def unregister(self, event_id, *, strict=False):
        if strict:
            self._event_ids.remove(event_id)
            return True

        was_present = event_id in self._event_ids
        self._event_ids.discard(event_id)
        return was_present

    def __contains__(self, event_id):
        return event_id in self._event_ids

    def __len__(self):
        return len(self._event_ids)

    def snapshot(self):
        return frozenset(self._event_ids)


registry = EventRegistry()
assert registry.register('evt-1') is True
assert registry.register('evt-1') is False
assert registry.register('evt-2') is True
assert len(registry) == 2
assert 'evt-1' in registry
assert registry.unregister('evt-missing') is False
assert registry.unregister('evt-1') is True
assert registry.snapshot() == frozenset({'evt-2'})
print(registry.snapshot())

frozenset({'evt-2'})


# Problem 19 — Set-backed FIFO queue

## Task

Build a FIFO queue that prevents the same item from being scheduled more than once at a time.

In [22]:
class UniqueFIFO:
    def __init__(self):
        self._queue = deque()
        self._scheduled = set()

    def push(self, item):
        if item in self._scheduled:
            return False
        self._scheduled.add(item)
        self._queue.append(item)
        return True

    def pop(self):
        if not self._queue:
            raise IndexError('pop from empty UniqueFIFO')
        item = self._queue.popleft()
        self._scheduled.remove(item)
        return item

    def __len__(self):
        return len(self._queue)


queue = UniqueFIFO()
assert queue.push('A') is True
assert queue.push('B') is True
assert queue.push('A') is False
assert queue.pop() == 'A'
assert queue.push('A') is True
assert queue.pop() == 'B'
assert queue.pop() == 'A'
assert len(queue) == 0
print('UniqueFIFO checks passed.')

UniqueFIFO checks passed.


# Problem 20 — Membership benchmark

## Task

Compare list and set membership for values near the start, near the end, and absent. Use repeated trials and report medians.

In [23]:
def benchmark_membership(size=30_000, number=1_000, repeats=5):
    values_list = list(range(size))
    values_set = set(values_list)
    cases = {'near_start': 3, 'near_end': size - 1, 'missing': -1}
    results = {}

    for case_name, target in cases.items():
        list_times = repeat(
            stmt='target in values_list',
            globals={'target': target, 'values_list': values_list},
            number=number,
            repeat=repeats,
        )
        set_times = repeat(
            stmt='target in values_set',
            globals={'target': target, 'values_set': values_set},
            number=number,
            repeat=repeats,
        )
        results[case_name] = {
            'list_seconds': median(list_times),
            'set_seconds': median(set_times),
        }

    return results


benchmark_results = benchmark_membership()
for case_name, timings in benchmark_results.items():
    ratio = timings['list_seconds'] / timings['set_seconds']
    print(
        f"{case_name:>10}: "
        f"list={timings['list_seconds']:.6f}s, "
        f"set={timings['set_seconds']:.6f}s, "
        f"list/set={ratio:.1f}x"
    )

near_start: list=0.000120s, set=0.000042s, list/set=2.8x
  near_end: list=0.440136s, set=0.000046s, list/set=9589.0x
   missing: list=0.240768s, set=0.000024s, list/set=9990.4x


List membership is linear in the worst case. Set membership is expected constant time on average, though hashing overhead can make tiny or early-hit list searches competitive.

# Problem 21 — Shallow memory comparison

## Task

Compare the shallow container sizes of lists, sets, and dictionaries.

In [24]:
def shallow_container_sizes(count):
    values_list = list(range(count))
    values_set = set(values_list)
    values_dict = dict.fromkeys(values_list)

    return {
        'count': count,
        'list_bytes': sys.getsizeof(values_list),
        'set_bytes': sys.getsizeof(values_set),
        'dict_bytes': sys.getsizeof(values_dict),
    }


for count in (0, 1, 10, 100, 1_000):
    print(shallow_container_sizes(count))

{'count': 0, 'list_bytes': 56, 'set_bytes': 216, 'dict_bytes': 64}
{'count': 1, 'list_bytes': 72, 'set_bytes': 216, 'dict_bytes': 224}
{'count': 10, 'list_bytes': 136, 'set_bytes': 728, 'dict_bytes': 352}
{'count': 100, 'list_bytes': 856, 'set_bytes': 8408, 'dict_bytes': 4688}
{'count': 1000, 'list_bytes': 8056, 'set_bytes': 32984, 'dict_bytes': 36952}


Container sizes grow in jumps because CPython over-allocates. Exact values are implementation-dependent and do not include all referenced objects.

# Problem 22 — Approximate deep size with cycle protection

## Task

Recursively estimate reachable memory while avoiding double-counting shared objects and infinite loops caused by cycles.

In [25]:
def approximate_deep_size(value, seen_ids=None):
    if seen_ids is None:
        seen_ids = set()

    object_id = id(value)
    if object_id in seen_ids:
        return 0

    seen_ids.add(object_id)
    size = sys.getsizeof(value)

    if isinstance(value, dict):
        size += sum(
            approximate_deep_size(key, seen_ids)
            + approximate_deep_size(item, seen_ids)
            for key, item in value.items()
        )
    elif isinstance(value, (list, tuple, set, frozenset, deque)):
        size += sum(approximate_deep_size(item, seen_ids) for item in value)

    return size


shared = 'a relatively long shared string'
example = [shared, shared, {shared}, {'key': shared}]
deep_size = approximate_deep_size(example)
assert deep_size >= sys.getsizeof(example)
print('Approximate deep size:', deep_size, 'bytes')

Approximate deep size: 604 bytes


This is educational, not a replacement for a production memory profiler.

# Problem 23 — Randomized property tests

## Task

Validate key set identities over many random inputs.

In [26]:
def check_set_identities(trials=500, universe=range(-20, 21), seed=7):
    rng = random.Random(seed)
    universe = list(universe)
    universe_set = set(universe)

    for _ in range(trials):
        left = {value for value in universe if rng.random() < 0.35}
        right = {value for value in universe if rng.random() < 0.35}
        third = {value for value in universe if rng.random() < 0.35}

        assert left | right == right | left
        assert left & right == right & left
        assert left ^ right == right ^ left
        assert left - right == left & (universe_set - right)
        assert left ^ right == (left - right) | (right - left)
        assert left & (right | third) == (left & right) | (left & third)
        assert left == (left & right) | (left - right)
        assert (left & right).isdisjoint(left - right)

    return True


assert check_set_identities()
print('All randomized set-algebra properties passed.')

All randomized set-algebra properties passed.


# Problem 24 — Capstone: role-based access evaluation

## Task

Combine permissions from roles, apply explicit grants and denials, reject unknown roles, report redundant grants, and verify required permissions.

In [27]:
def evaluate_access(
    assigned_roles,
    role_permissions,
    *,
    explicit_grants=frozenset(),
    explicit_denials=frozenset(),
    required_permissions=frozenset(),
):
    assigned_roles = set(assigned_roles)
    known_roles = set(role_permissions)

    unknown_roles = assigned_roles - known_roles
    if unknown_roles:
        raise ValueError(f'Unknown roles: {sorted(unknown_roles)}')

    role_grants = (
        set().union(*(set(role_permissions[role]) for role in assigned_roles))
        if assigned_roles
        else set()
    )

    explicit_grants = set(explicit_grants)
    explicit_denials = set(explicit_denials)
    required_permissions = set(required_permissions)

    effective = (role_grants | explicit_grants) - explicit_denials
    missing_required = required_permissions - effective

    return {
        'effective': frozenset(effective),
        'role_grants': frozenset(role_grants),
        'redundant_grants': explicit_grants & role_grants,
        'denied_but_granted': explicit_denials & (role_grants | explicit_grants),
        'missing_required': missing_required,
        'valid': not missing_required,
    }


role_permissions = {
    'viewer': {'read'},
    'editor': {'read', 'write'},
    'auditor': {'read', 'export'},
    'administrator': {'read', 'write', 'export', 'delete'},
}

access = evaluate_access(
    {'editor', 'auditor'},
    role_permissions,
    explicit_grants={'export', 'comment'},
    explicit_denials={'write'},
    required_permissions={'read', 'comment'},
)

assert access['effective'] == frozenset({'read', 'export', 'comment'})
assert access['redundant_grants'] == {'export'}
assert access['denied_but_granted'] == {'write'}
assert access['missing_required'] == set()
assert access['valid'] is True
print(access)

{'effective': frozenset({'read', 'export', 'comment'}), 'role_grants': frozenset({'read', 'export', 'write'}), 'redundant_grants': {'export'}, 'denied_but_granted': {'write'}, 'missing_required': set(), 'valid': True}


# Additional challenge exercises

1. Compute Jaccard similarity and define the result for two empty sets.
2. Given daily visitor-ID sets, find visitors present every day, exactly one day, and at least two days.
3. Build a case-insensitive username registry that preserves original spelling.
4. Determine course eligibility from prerequisite sets.
5. Detect similar documents using word-shingle sets and Jaccard similarity.
6. Extend `UniqueFIFO` with `discard(item)` while keeping both structures consistent.
7. Return counterexamples explaining why one set is not a subset of another.
8. Benchmark `set(values)`, `unique_in_order(values)`, and `dict.fromkeys(values)`.
9. Build a reversible freezer that preserves original nested container types.
10. Implement a dependency resolver that uses sets for completed, pending, and blocked tasks.

# Summary

Use sets when the problem is fundamentally about uniqueness, fast membership, overlap, exclusion, or unordered combination.

Choose another structure when you need stable ordering, positional access, duplicates, priority, or key-to-value mapping. Strong solutions often combine a set with a list, dictionary, deque, or heap.